# 논문용 MVTec AD 15-category artifact 생성

선택한 category를 독립 학습하고 검증 가능한 Jetson bundle을 Google Drive에 저장합니다. Colab GPU 세션이 끊기면 같은 `RUN_NAME`으로 다시 실행하세요. 이미 검증된 category bundle은 건너뜁니다. 한 세션에 전부 끝내기 어렵다면 `CATEGORIES_TO_RUN`을 3~5개씩 나누세요.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys

REPOSITORY_URL = "https://github.com/minwoo1119/edge-anomaly-detection.git"
REPOSITORY_ROOT = Path("/content/edge-anomaly-detection")
DRIVE_INPUT_ROOT = Path("/content/drive/MyDrive/edge-anomaly-input")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/edge-anomaly-output")
ALL_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather",
    "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper",
]
CATEGORIES_TO_RUN = ALL_CATEGORIES
RUN_NAME = "mvtec15_seed42_c01"
SEED = 42
CORESET_RATIO = "0.10"
CALIBRATION_IMAGES = 100
AUTO_DOWNLOAD_DATASET = True


In [ ]:
import torch
from google.colab import drive
drive.mount("/content/drive")
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0))

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPOSITORY_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPOSITORY_ROOT / "requirements.txt")], check=True)
subprocess.run(["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], check=True)


In [ ]:
local_dataset_root = REPOSITORY_ROOT / "datasets" / "MVTecAD"
local_model_root = REPOSITORY_ROOT / "models" / RUN_NAME
local_metadata_root = REPOSITORY_ROOT / "results" / "metadata" / RUN_NAME
local_output_root = Path("/content") / RUN_NAME
drive_run_root = DRIVE_OUTPUT_ROOT / RUN_NAME
drive_run_root.mkdir(parents=True, exist_ok=True)

for category in CATEGORIES_TO_RUN:
    if category not in ALL_CATEGORIES:
        raise ValueError(f"Unknown MVTec category: {category}")
    destination = drive_run_root / category
    complete_marker = destination / "COMPLETE.json"
    if complete_marker.is_file():
        print(f"SKIP complete: {category}")
        continue

    source_category = DRIVE_INPUT_ROOT / "datasets" / "MVTecAD" / category
    local_category = local_dataset_root / category
    if not local_category.is_dir() and source_category.is_dir():
        local_category.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_category, local_category)
    if not local_category.is_dir() and AUTO_DOWNLOAD_DATASET:
        local_dataset_root.mkdir(parents=True, exist_ok=True)
        download = (
            "from anomalib.data import MVTecAD; "
            f"d=MVTecAD(root={str(local_dataset_root)!r}, category={category!r}); d.prepare_data()"
        )
        subprocess.run([sys.executable, "-c", download], check=True, cwd=REPOSITORY_ROOT)
    required = [local_category / "train" / "good", local_category / "test" / "good", local_category / "ground_truth"]
    if not all(path.is_dir() for path in required):
        raise FileNotFoundError(f"Incomplete dataset for {category}: {required}")

    checkpoint = local_model_root / f"patchcore_{category}.ckpt"
    training_manifest = local_metadata_root / f"train_{category}.json"
    if not checkpoint.is_file() or not training_manifest.is_file():
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        training_manifest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            sys.executable, "src/train.py", "--dataset-root", str(local_dataset_root),
            "--category", category, "--checkpoint", str(checkpoint),
            "--manifest", str(training_manifest), "--coreset-ratio", CORESET_RATIO,
            "--accelerator", "gpu", "--devices", "1", "--seed", str(SEED),
        ], check=True, cwd=REPOSITORY_ROOT)

    output_root = local_output_root / category
    if output_root.exists():
        shutil.rmtree(output_root)
    subprocess.run([
        sys.executable, "src/run_colab_pipeline.py", "--checkpoint", str(checkpoint),
        "--training-manifest", str(training_manifest), "--dataset-root", str(local_dataset_root),
        "--category", category, "--output-root", str(output_root),
        "--calibration-images", str(CALIBRATION_IMAGES), "--device", "cuda",
        "--export-device", "cpu",
    ], check=True, cwd=REPOSITORY_ROOT)

    bundle = output_root / f"{category}_artifacts.zip"
    checksum = bundle.with_suffix(bundle.suffix + ".sha256")
    subprocess.run([sys.executable, "src/verify_artifact_bundle.py", "--bundle", str(bundle)], check=True, cwd=REPOSITORY_ROOT)
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    shutil.copy2(bundle, destination / bundle.name)
    shutil.copy2(checksum, destination / checksum.name)
    marker = {
        "category": category, "coreset_ratio": float(CORESET_RATIO), "seed": SEED,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "git_commit": subprocess.check_output(["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"], text=True).strip(),
    }
    complete_marker.write_text(json.dumps(marker, indent=2) + "\n")
    print(f"COMPLETE: {category} -> {destination}")


In [ ]:
completed = sorted(path.parent.name for path in drive_run_root.glob("*/COMPLETE.json"))
missing = [category for category in ALL_CATEGORIES if category not in completed]
print("completed:", completed)
print("missing:", missing)
print("Drive output:", drive_run_root)
